In [ ]:
# Configuration
INPUT_H5AD = "/home/h2048/data/py/1128/bbknn_celltype_analysis/T/adata_T_bbknn.h5ad"
OUTPUT_DIR = "/home/h2048/data/py/1203/scvi_test_T"

# scVI parameters
SCVI_N_LATENT = 30        # Latent dimensions (default=10, use 30 for complex datasets)
SCVI_N_LAYERS = 2         # Neural network layers
SCVI_DROPOUT = 0.1        # Dropout rate
SCVI_MAX_EPOCHS = 400     # Training epochs (will auto-stop if converged)

# UMAP parameters (for comparison with BBKNN)
UMAP_MIN_DIST = 0.15
UMAP_SPREAD = 1.3

# Clustering
LEIDEN_RESOLUTION = 3.0

# Create output
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
fig_dir = output_dir / "figures"
fig_dir.mkdir(exist_ok=True)

print(f"Output directory: {output_dir}")

---
## 1. Load and Preprocess Data

In [ ]:
# Load data
print("Loading data...")
adata = sc.read_h5ad(INPUT_H5AD)
print(f"Loaded: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")

# Check batch info
n_batches = adata.obs['dataset'].nunique()
print(f"\nNumber of batches: {n_batches}")
print(adata.obs['dataset'].value_counts().head(10))

# Check if tissue info available
if 'tissue' in adata.obs.columns:
    print(f"\nTissue types: {adata.obs['tissue'].unique()}")
    has_tissue = True
else:
    print("\nNo 'tissue' column found")
    has_tissue = False

In [ ]:
# Preprocessing for scVI
print("\nPreprocessing for scVI...")

# Store raw counts if available
if adata.raw is None and 'counts' in adata.layers:
    adata.layers['counts'] = adata.layers['counts'].copy()
elif adata.raw is None:
    print("Warning: No raw counts found, using current .X")
    adata.layers['counts'] = adata.X.copy()
else:
    adata.layers['counts'] = adata.raw.X.copy()

# Store original for comparison
adata_original = adata.copy()

# Basic filtering
sc.pp.filter_genes(adata, min_cells=10)
print(f"After filtering: {adata.shape[1]:,} genes")

# Select highly variable genes (scVI works best with HVGs)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=4000,
    flavor='seurat_v3',
    batch_key='dataset',  # Account for batch when selecting HVGs
    subset=True
)

print(f"Selected HVGs: {adata.shape[1]:,}")
print(f"Final dataset: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")

---
## 2. Setup scVI Model

In [ ]:
# Setup AnnData for scVI
print("\nSetting up scVI...")

# Setup with batch key and optional confounder
if has_tissue:
    print("Using 'tissue' as categorical covariate (to preserve biological variation)")
    scvi.model.SCVI.setup_anndata(
        adata,
        layer='counts',
        batch_key='dataset',
        categorical_covariate_keys=['tissue']  # Preserve tissue differences
    )
else:
    print("Using only 'dataset' as batch key")
    scvi.model.SCVI.setup_anndata(
        adata,
        layer='counts',
        batch_key='dataset'
    )

print("✓ AnnData setup complete")

In [ ]:
# Create scVI model
print("\nCreating scVI model...")

vae = scvi.model.SCVI(
    adata,
    n_latent=SCVI_N_LATENT,
    n_layers=SCVI_N_LAYERS,
    dropout_rate=SCVI_DROPOUT,
    gene_likelihood='nb'  # Negative binomial for count data
)

print(f"Model parameters:")
print(f"  Latent dimensions: {SCVI_N_LATENT}")
print(f"  Neural network layers: {SCVI_N_LAYERS}")
print(f"  Dropout rate: {SCVI_DROPOUT}")
print(f"  Gene likelihood: negative binomial")
print(f"\n✓ Model created")

---
## 3. Train scVI Model

This may take 10-30 minutes depending on your hardware

In [ ]:
import time

print("\nTraining scVI model...")
print(f"Max epochs: {SCVI_MAX_EPOCHS}")
print("Training will auto-stop if model converges early\n")

start_time = time.time()

# Train with early stopping
vae.train(
    max_epochs=SCVI_MAX_EPOCHS,
    early_stopping=True,
    early_stopping_patience=15,
    plan_kwargs={'lr': 1e-3}  # Learning rate
)

elapsed = time.time() - start_time
print(f"\n✓ Training completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")

# Plot training history
train_history = vae.history['elbo_train']
val_history = vae.history.get('elbo_validation', [])

plt.figure(figsize=(8, 4))
plt.plot(train_history, label='Training ELBO', alpha=0.7)
if len(val_history) > 0:
    plt.plot(val_history, label='Validation ELBO', alpha=0.7)
plt.xlabel('Epoch')
plt.ylabel('ELBO (Evidence Lower Bound)')
plt.title('scVI Training History')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(fig_dir / 'scvi_training_history.pdf', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Training history plot saved")

---
## 4. Extract Latent Representation

In [ ]:
# Get latent representation (batch-corrected embedding)
print("\nExtracting latent representation...")
latent = vae.get_latent_representation()

# Store in adata
adata.obsm['X_scvi'] = latent

print(f"Latent representation shape: {latent.shape}")
print(f"Stored in adata.obsm['X_scvi']")

# Save model (optional, in case you want to load it later)
model_dir = output_dir / "scvi_model"
vae.save(model_dir, overwrite=True)
print(f"\n✓ Model saved to: {model_dir}")

---
## 5. Compute Neighbors, UMAP, and Clustering

In [ ]:
# Compute neighbors on scVI latent space
print("\nComputing neighbors on scVI latent space...")
sc.pp.neighbors(adata, use_rep='X_scvi', n_neighbors=30)
print("✓ Neighbors computed")

# UMAP
print("\nComputing UMAP...")
sc.tl.umap(adata, min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD)
print(f"✓ UMAP completed (min_dist={UMAP_MIN_DIST}, spread={UMAP_SPREAD})")

# Clustering at multiple resolutions
resolutions = [2.0, 2.5, 3.0, 3.5]
print("\nClustering at multiple resolutions...")
for res in resolutions:
    sc.tl.leiden(adata, resolution=res, key_added=f'leiden_scvi_res{res}')
    n_clusters = adata.obs[f'leiden_scvi_res{res}'].nunique()
    print(f"  Resolution {res}: {n_clusters} clusters")

# Set default clustering
adata.obs['leiden_scvi'] = adata.obs[f'leiden_scvi_res{LEIDEN_RESOLUTION}']
n_clusters_default = adata.obs['leiden_scvi'].nunique()
print(f"\n✓ Default clustering: resolution={LEIDEN_RESOLUTION}, n_clusters={n_clusters_default}")

---
## 6. Calculate Batch Mixing Metrics

In [ ]:
from scipy.stats import entropy
from sklearn.neighbors import NearestNeighbors

def calculate_batch_mixing(adata, use_rep='X_scvi', batch_key='dataset', n_neighbors=50):
    """Calculate batch mixing entropy"""
    knn = NearestNeighbors(n_neighbors=n_neighbors)
    knn.fit(adata.obsm[use_rep])
    _, indices = knn.kneighbors(adata.obsm[use_rep])
    
    entropies = []
    for idx in indices:
        batch_dist = adata.obs[batch_key].iloc[idx].value_counts(normalize=True)
        entropies.append(entropy(batch_dist))
    
    return np.mean(entropies), np.std(entropies)

# Calculate metrics
print("\nCalculating batch mixing metrics...")
mean_entropy, std_entropy = calculate_batch_mixing(adata, use_rep='X_scvi')

print(f"\nBatch Mixing Entropy:")
print(f"  scVI: {mean_entropy:.3f} ± {std_entropy:.3f}")
print(f"\nComparison:")
print(f"  BBKNN (from previous test): 0.155 ± 0.301")
print(f"  Ideal range: 2.0 - 2.5")
print(f"  Maximum (uniform): {np.log(n_batches):.3f}")

improvement = (mean_entropy - 0.155) / (np.log(n_batches) - 0.155) * 100
print(f"\n  Improvement over BBKNN: {improvement:.1f}% toward ideal mixing")

---
## 7. Visualization - scVI Results

In [ ]:
# UMAP colored by batch
print("\nGenerating UMAP plots...")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Batch
sc.pl.umap(adata, color='dataset', ax=axes[0], 
           title='scVI: Batch Mixing', show=False, legend_loc='right margin')

# Clustering
sc.pl.umap(adata, color='leiden_scvi', ax=axes[1],
           title=f'scVI: Leiden Clustering (n={n_clusters_default})',
           show=False, legend_loc='on data', legend_fontsize=6)

plt.tight_layout()
plt.savefig(fig_dir / 'scvi_umap_batch_cluster.pdf', dpi=150, bbox_inches='tight')
plt.show()

print("✓ UMAP plots saved")

In [ ]:
# Marker genes
key_markers = ['NCAM1', 'CD4', 'CD8A', 'CD3D', 'GZMB', 'FOXP3']
available_markers = [m for m in key_markers if m in adata_original.var_names]

if len(available_markers) > 0:
    print(f"\nPlotting marker genes: {available_markers}")
    
    # Transfer raw counts for marker visualization
    adata.raw = adata_original[adata.obs_names, :].copy()
    
    fig = sc.pl.umap(adata, color=available_markers[:6], 
                     use_raw=True, ncols=3, vmax='p99', 
                     cmap='Reds', return_fig=True, show=False)
    fig.savefig(fig_dir / 'scvi_umap_markers.pdf', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✓ Marker gene plots saved")
else:
    print("\nWarning: No key markers found in dataset")

---
## 8. Multi-Resolution Clustering Comparison

In [ ]:
# Compare clustering at different resolutions
cluster_keys = [f'leiden_scvi_res{res}' for res in resolutions]

fig = sc.pl.umap(adata, color=cluster_keys, ncols=2,
                 return_fig=True, show=False, legend_loc='on data',
                 legend_fontsize=6)
fig.savefig(fig_dir / 'scvi_umap_multiresolution.pdf', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Multi-resolution comparison saved")

---
## 9. Quality Assessment

In [ ]:
# Check batch distribution per cluster
print("\nBatch distribution analysis...")
batch_cluster = pd.crosstab(
    adata.obs['leiden_scvi'],
    adata.obs['dataset'],
    normalize='index'
)

# Plot heatmap
plt.figure(figsize=(12, max(6, n_clusters_default * 0.3)))
sns.heatmap(batch_cluster, cmap='viridis', cbar_kws={'label': 'Proportion'})
plt.xlabel('Dataset (Batch)')
plt.ylabel('Cluster')
plt.title('Batch Distribution per Cluster\n(Uniform distribution indicates good mixing)')
plt.tight_layout()
plt.savefig(fig_dir / 'scvi_batch_per_cluster.pdf', dpi=150, bbox_inches='tight')
plt.show()

# Find batch-specific clusters (potential over/under-correction)
batch_dominant = (batch_cluster > 0.5).sum(axis=1)
problematic_clusters = batch_dominant[batch_dominant > 0].index.tolist()

if len(problematic_clusters) > 0:
    print(f"\n⚠️  Clusters dominated by single batch: {problematic_clusters}")
    print("   This might indicate under-correction or true biological differences")
else:
    print("\n✓ No clusters dominated by single batch (good batch mixing)")

---
## 11. Save Results

In [ ]:
# Save scVI-integrated data
output_file = output_dir / 'adata_tcell_scvi_integrated.h5ad'
adata.write_h5ad(output_file)
print(f"\n✓ Integrated data saved to: {output_file}")

# Save summary metrics
summary = {
    'method': 'scVI',
    'n_cells': adata.n_obs,
    'n_genes': adata.n_vars,
    'n_batches': n_batches,
    'n_latent': SCVI_N_LATENT,
    'resolution': LEIDEN_RESOLUTION,
    'n_clusters': n_clusters_default,
    'batch_mixing_entropy': mean_entropy,
    'batch_mixing_std': std_entropy,
    'umap_min_dist': UMAP_MIN_DIST,
    'umap_spread': UMAP_SPREAD
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(output_dir / 'scvi_summary.csv', index=False)
print(f"✓ Summary saved to: {output_dir / 'scvi_summary.csv'}")

---
## 12. Summary and Recommendations

In [ ]:
print("\n" + "="*70)
print("scVI INTEGRATION TEST - SUMMARY")
print("="*70)

print("\n[ Model Configuration ]")
print(f"  Latent dimensions: {SCVI_N_LATENT}")
print(f"  Training epochs: {len(train_history)}")
print(f"  Final ELBO: {train_history[-1]:.2f}")

print("\n[ Batch Mixing ]")
print(f"  BBKNN entropy: 0.155 (very poor)")
print(f"  scVI entropy: {mean_entropy:.3f} ({improvement:.0f}% improvement)")
print(f"  Ideal range: 2.0 - 2.5")

if mean_entropy > 1.5:
    print("  ✅ Good batch mixing achieved")
elif mean_entropy > 0.8:
    print("  ⚠️  Moderate improvement, consider increasing n_latent or epochs")
else:
    print("  ❌ Limited improvement, batch effects may be very strong")

print(f"\n[ Clustering ]")
print(f"  Resolution: {LEIDEN_RESOLUTION}")
print(f"  Number of clusters: {n_clusters_default}")
if len(problematic_clusters) > 0:
    print(f"  ⚠️  {len(problematic_clusters)} clusters dominated by single batch")
else:
    print(f"  ✅ No batch-dominated clusters")

print("\n[ Next Steps ]")
print("  1. Compare UMAP plots:")
print("     - Are NK/CD4/CD8 better separated than BBKNN?")
print("     - Are cluster boundaries smoother?")
print("     - Is batch mixing improved?")
print("  2. Check marker gene expression:")
print("     - NCAM1, CD4, CD8A patterns match expected biology?")
print("  3. If results are better:")
print("     - Use scVI latent space for downstream analysis")
print("     - Can combine with UMAP parameter optimization")
print("  4. If results are similar or worse:")
print("     - Stick with BBKNN (simpler, faster)")
print("     - Or try scVI with different parameters")

print("\n[ Output Files ]")
print(f"  Integrated data: {output_file}")
print(f"  Model checkpoint: {model_dir}")
print(f"  Figures: {fig_dir}/*.pdf")
print(f"  Summary: {output_dir / 'scvi_summary.csv'}")

print("\n" + "="*70)
print("ANALYSIS COMPLETE")
print("="*70)